In [4]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import metrics
from knn_model_optimized import KNN_model

from scipy.sparse import load_npz

### 1) Análisis del Modelo Base (Baseline Model):

In [ ]:
# ...

### 2) Análisis del Modelo KNN: 

In [6]:
path = '../data/processed/'

train_set = load_npz(path+'train_set.npz')
test_set = load_npz(path+'test_set.npz')

max_users = 500
k = 20
knn_model = KNN_model(train_set, test_set, k=k, eval_max_users=max_users)

knn_model.evaluate()

Evaluando subset: 500 usuarios, 14768 interacciones


{'mae': np.float64(1.198329703743629),
 'rmse': np.float64(1.5071126813158737),
 'precision@10': np.float64(0.08360655737704918),
 'recall@10': np.float64(0.4514441842310695),
 'f1@10': np.float64(0.14108454079704566),
 'ndcg@10': np.float64(0.44409516941829974)}

#### **Métricas de error**: 
Al medir el error en una escala acotada de 1 a 5, podemos evaluar la desviación de forma absoluta sobre el rango del 
problema: 

- **MAE** (1.198 $\approx$ 1.2 estrellas): En promedio, cuando el modelo intenta predecir cuántas estrellas le daría un usuario a una canción, se desvía en poco más de 1 estrella. Si la valoración real en test es un 4 (alta preferencia), el modelo suele estimar un 2.8 o un 5.2 (fuera de rango, truncado a 5).

- **RMSE** (1.507 $\approx$ 1.5 estrellas): La diferencia sustancial entre el MAE y el RMSE (0.3 estrellas) confirma matemáticamente que el modelo comete errores severos en casos específicos. 

Este comportamiento es un síntoma típico del algoritmo de *deviation from mean* combinado con el sesgo que introduce tu mapeo. Al transformar playcounts a escala 1-5, es probable que las canciones que se escuchan de forma masiva (los hits) se hayan convertido en "5s" absolutos. Si un vecino comparte el gusto por la canción pero tiene un historial de reproducción general muy bajo, su predicción se verá arrastrada hacia abajo por su propia media, provocando que el modelo penalice el hit y prediga, por ejemplo, un 2 o un 3. Ese desfase de 2 o 3 estrellas en los extremos es lo que dispara el RMSE a 1.5. De este modo, el KNN tiende a la **baja varianza** (predice muchos valores cercanos a la media general) y sufre con los valores extremos (los 1s y los 5s puros), mostrando **falta de capacidad de generalización del KNN ante perfiles estos perfiles extremos**, lo que explica ese RMSE de 1.5. 

#### **Métricas de recomendación y ranking**: 

- **Precision@10** (8.36%): De las 10 canciones que se recomienda con mayor puntuación predicha, menos de una aparece en el conjunto de test. En una escala 1-5, esto nos dice que el algoritmo tiende a inflar las predicciones de muchos ítems (generando falsos positivos masivos) o que el set de test padece de una alta escasez de datos (**sparsity**), algo que hemos verificado que es así. Como la métrica asume que **si una recomendación no está en el test (algo que sucede en numerosas ocasiones por el sparsity) es un "voto negativo"**, pese a que tal vez sea una recomendación muy buena que al usuario le encantaría (pero que no sabemos porque no ha escuchado esa canción), la precision disminuye considerablemente. 

- **Recall@10** (45.14%): A pesar de la baja precisión, el modelo es capaz de encontrar el 45% de las canciones que el usuario realmente valoró en el test. Nuestro modelo de KNN es un **excelente "buscador" de tendencias generales**. Sabe en qué vecindario musical se mueve el usuario, capturando casi la mitad de sus preferencias reales. Sin embargo, esto, de nuevo debido al **sparsity**, puede estar inflado al haber escaso volumen de canciones reales en el set de test por usuario. 

- **NDCG@10** (44.41%): Un 44.4% en una escala 1-5 es una métrica muy sólida. Significa que el **orden posicional es correcto**. Cuando el modelo acierta una canción que el usuario valoró con un 4 o un 5 en el set de test, el algoritmo ha sido capaz de **colocarla en los primeros puestos** de la lista de recomendación (puestos 1, 2 o 3).

#### **Conclusiones del análisis:**
La conclusión principal gira en torno a un concepto clave: **el modelo es un excelente clasificador de tendencias (Ranking), pero un predictor impreciso de notas (Regresión).**

1) Nuestro sistema actual sufre del **sesgo de centralidad de los modelos de vecindario**. Al promediar las desviaciones de los $K$ vecinos, el modelo tiende a suavizar las predicciones hacia la media general. Por eso el MAE se queda estancado en ~1.2 estrellas: al modelo le cuesta predecir los "1s" puros o los "5s" puros si los vecinos no son unánimes.

2) La combinación de baja Precisión (8%) con alto Recall (45%) y buen NDCG (44%) dibuja un recomendador con un perfil muy específico: un **motor de descubrimiento**. El sistema le propone al usuario canciones que matemáticamente tienen altas probabilidades de gustarle (alto Recall). El hecho de que la precisión sea baja es un reflejo de que el usuario no ha tenido la oportunidad física de escuchar y valorar esas canciones en el set de test. Sin embargo, para los elementos que la validación sí puede comprobar, la experiencia de usuario es óptima porque los temas más relevantes aparecen arriba del todo de la lista gracias al NDCG.